
## Objetivo del programa
#### Descargar productos maritimos Sentinel 3 OLCI desde la plataforma Openeo del Copernicus Data Space Ecosystem. El programa entrega los promedios Semanales de los prpoductos datellados
## Productos a descargar
#### -***TSM_NN***: Total Suspended matter
#### -***CHL_NN***: Concetración total de clorofila
#### -***KD490_M07***: Coeficiente de atenuación difusa a los 490 nm. Indicador de la turbidez
#### -***ADG443_NN***: Coeficiente de absorción a los 443 nm. Indicador de la luz absorvida por el CDOM

#### Formato de descarga
##### -***Geotiff***

#### Georreferencia
##### -***EPSG:4326***


In [ ]:
import openeo
import time
from datetime import datetime, timedelta
from openeo.processes import process, mean  # Importar procesos necesarios
# Crear instancias del proceso (NO se llaman directamente)
bitwise_and = process("bitwise_and")
eq = process("eq")
logical_and = process("and")

import os
import shutil

# --- Paso 1: Conexión y autenticación ---
conn = openeo.connect("https://openeo.dataspace.copernicus.eu").authenticate_oidc()
print("✅ Autenticación exitosa")

# --- Paso 2: Parámetros generales ---
spatial_extent = {
    "west": -65.5,
    "south": -43,
    "east": -62.5,
    "north": -40.5
}
year = 2024 # Año de procesamiento: se descargan los promedios semanales para todo el año especificado
product = "ADG443_NN" # Producto

# Crear carpeta si no existe
output_folder = r"" # Especificar Folder Local para la descarga
os.makedirs(output_folder, exist_ok=True)

# --- Paso 3: Definir fechas semanales ---
start = datetime(year, 1, 1)
delta = timedelta(days=7)
week = 0

while week < 52:

    # Obtengo la fecha del principio de la semana
    start_date = start + (delta * week )
    
    # Obtengo le fecha del final se la semana
    if week == 51:
        end_date = datetime(year + 1, 1, 1) - timedelta(days=1) # Al principio del año siguiente le resto 1 día
    else:
        end_date = start_date + delta - timedelta(days=1)
        
    print(f"📅 Procesando semana{week}: {start_date.date()} a {end_date.date()}")
    
    # --- Paso 4: Cargar colección y calcular el promedio temporal ---

    cube = conn.load_collection(
        "SENTINEL3_OLCI_L2_WATER",
        spatial_extent=spatial_extent,
        temporal_extent=[start_date, end_date],
        bands=[product, "FLAGS"]
    )

    # --- Separar los datos de la máscara ---
    data = cube.band(product)
    flags = cube.band("FLAGS")

    # --- Aplicar la máscara de FLAGS ---
    good = ((flags & 1087) == 0)  # Bits 1-1024 (1+2+4+8+16+32+1024=1087)

    # Aplicar máscara
    masked_data = data.mask(good)
    
    
    # --- especificar el promedio temporal ---
    weekly_mean = masked_data.reduce_dimension(dimension="t", reducer=mean)
    result = weekly_mean.save_result(format="GTIFF")
    
    # ---Paso 5: Crear  y ejecutar el Job --- #
    job_title = f"{product}_{start_date.date().isoformat()}"
    job = result.create_job(title=job_title)
    job_info = job.describe_job()
    print(job_info)
    job.start_and_wait()
    results = job.get_results()

    # --- Paso 6: Descargar en carpeta temporal y renombrar el archivo ---
    temp_folder = os.path.join(output_folder, f"temp_{start_date.date().isoformat()}")
    os.makedirs(temp_folder, exist_ok=True)
    print(f"Este es el temp folder: {temp_folder}")
    results.download_files(target=temp_folder)

    # ---Paso 7: Renombrar archivo principal y moverlo ---
    for file in os.listdir(temp_folder):
        if file.endswith(".tif"):
            old_path = os.path.join(temp_folder, file)
            new_name = f"{product}_{start_date.date().isoformat()}.tif"
            new_path = os.path.join(output_folder, new_name)
            shutil.copy(old_path, new_path)
            print(f"✅ Guardado: {new_name}")

   

    # --- Próxima semana ---
    week += 1
    time.sleep(5) # Pausa de 5 segundos antes de empezar el sigiuente request

print("procesamiento terminado con exito")

Authenticated using refresh token.
✅ Autenticación exitosa
📅 Procesando semana0: 2024-01-01 a 2024-01-07
{'created': '2025-08-08T13:04:17Z', 'id': 'j-25080813041747638618304454479f7c', 'process': {'process_graph': {'loadcollection1': {'arguments': {'bands': ['ADG443_NN', 'FLAGS'], 'id': 'SENTINEL3_OLCI_L2_WATER', 'spatial_extent': {'east': -62.5, 'north': -40.5, 'south': -43, 'west': -65.5}, 'temporal_extent': ['2024-01-01T00:00:00Z', '2024-01-07T00:00:00Z']}, 'process_id': 'load_collection'}, 'mask1': {'arguments': {'data': {'from_node': 'reducedimension1'}, 'mask': {'from_node': 'reducedimension2'}}, 'process_id': 'mask'}, 'reducedimension1': {'arguments': {'data': {'from_node': 'loadcollection1'}, 'dimension': 'bands', 'reducer': {'process_graph': {'arrayelement1': {'arguments': {'data': {'from_parameter': 'data'}, 'index': 0}, 'process_id': 'array_element', 'result': True}}}}, 'process_id': 'reduce_dimension'}, 'reducedimension2': {'arguments': {'data': {'from_node': 'loadcollectio